In [11]:
!pip install langchain
!pip install langchain-community
!pip install langchain-groq
!pip install langchain-tavily
!pip install langgraph
!pip install chromadb
!pip install sentence-transformers
!pip install PyMuPDF
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


...


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


# Agentic RAG Tanulást Segítő Alkalmazás

## Áttekintés
Ez a notebook egy **Agentic RAG (Retrieval-Augmented Generation)** alapú tanulást segítő chatbot prototípusa.

A felhasználó PDF dokumentumokat tölthet fel tárgyak szerint rendezve (pl. `docs/Matematika/`, `docs/Biologia/`), és természetes nyelven kérdezhet, összefoglalót vagy tanulási tervet kérhet belőlük.

## Technológiák
- **LangGraph** – agentic pipeline, state management
- **ChromaDB** – vektoros adatbázis, perzisztens tárolás
- **Groq API** – ingyenes LLM hívások (pl. llama-3.3-70b-versatile)
- **Sentence Transformers** – multilinguális embedding
- **Tavily** – web search fallback ha a dokumentumokban nincs elég információ

## Mappa struktúra
```
project/
├── docs/
│   ├── Matematika/      ← tárgy neve
│   │   └── tankönyv.pdf
│   └── Biologia/
│       └── biologia.pdf
├── vectorstore/         ← ChromaDB
├── .env                 ← API kulcsok
└── notebook.ipynb
```

## Szükséges API kulcsok (.env fájlban)
```
GROQ_API_KEY=gsk_...       
TAVILY_API_KEY=tvly-...    
```

---

## 1. Környezet inicializálása
Betölti a `.env` fájlt, létrehozza, betölti a szükséges mappákat, és listázza az elérhető tárgyakat.

In [12]:
import os
from dotenv import load_dotenv
from pathlib import Path


load_dotenv()

BASE_DIR = Path(".")
VECTOR_DB_DIR = BASE_DIR / "vectorstore"
DOCS_DIR = BASE_DIR / "docs"

VECTOR_DB_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

print(f"Docs mappa: {DOCS_DIR.resolve()}")
print(f"VectorDB mappa: {VECTOR_DB_DIR.resolve()}")

subjects = [f.name for f in DOCS_DIR.iterdir() if f.is_dir()]
print(f"Talált tárgyak: {subjects if subjects else 'Még nincs tárgy mappa'}")


Docs mappa: C:\Users\czove\Documents\GitHub\RAG_app\docs
VectorDB mappa: C:\Users\czove\Documents\GitHub\RAG_app\vectorstore
Talált tárgyak: ['Biologia', 'Matematika']


## LLM inicializálás
- API: groq. Limitáltan de ingyenesen használható, több modell is elérhető
- Ha nincs key(vagy error) akkor a mock llm választ adja vissza

In [13]:
from langchain_groq import ChatGroq

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
def mock_llm(prompt: str) -> str:
    return "[MOCK VÁLASZ] Ez egy szimulált LLM válasz fejlesztési célra."
if GROQ_API_KEY:
    llm = ChatGroq(
        model="openai/gpt-oss-20b",
        api_key=GROQ_API_KEY,
        temperature=0.7,
    )
    try:
        test = llm.invoke("Mondj egy mondatot magyarul.")
        print(f"✅ Groq kapcsolat OK: {test.content[:60]}...")
    except Exception as e:
        print(f"Groq nem elérhető, mock módban fut. Hiba: {e}")
        llm = None
else:
    print("GROQ_API_KEY nem található .env fájlban, mock módban fut.")
    llm = None

def invoke_llm(prompt: str) -> str:
    if llm:
        try:
            return llm.invoke(prompt).content
        except Exception as e:
            print(f"LLM hiba: {e}")
    return mock_llm(prompt)



✅ Groq kapcsolat OK: A napfényes napra szép, hosszú sétát javaslok a közeli erdőb...


## Embedding modell inicializálás és vector adatbázis létrehozása
### Embedding model: paraphrase-multilingual-MiniLM-L12-v2:
    - Ingyenes
    - Lokálisan fut
    - Kis méret de hatékony szemantikus kereséshez
    - Támogatja a magyar nyelvet
    
### ChromaDB:
    - Alkalmas a modell és adatbázis együttes kezelésére
    - Metaadat szűrés tárgyak szerint
    - HNSW-vel gyors keresés


In [14]:
from sentence_transformers import SentenceTransformer
import chromadb

print("Embedding modell betöltése...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Embedding modell kész.")

chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
existing = [c.name for c in chroma_client.list_collections()]
print(f"Meglévő kollekciók: {existing if existing else 'Még nincs indexelt tárgy'}")

print("\nSetup kész.")


Embedding modell betöltése...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding modell kész.
Meglévő kollekciók: ['Matematika', 'Biologia']

Setup kész.


## Fileok hashelése

### Felesleges újraolvasás, vektorizálás elkerülése
    1. Indexeléskor minden PDF-hez kiszámítja az aktuális hash-t
    2. Összehasonlítja a mentett hash-sel
    3. Ha egyezik → fájl nem változott, kihagyja
    4. Ha eltér vagy nincs mentve → újraindexeli

In [15]:
import hashlib
import json
import fitz  # PyMuPDF
from pathlib import Path


HASH_FILE = BASE_DIR / "vectorstore" / "file_hashes.json"

def load_hashes() -> dict:
    if HASH_FILE.exists():
        with open(HASH_FILE, "r") as f:
            return json.load(f)
    return {}

def save_hashes(hashes: dict):
    with open(HASH_FILE, "w") as f:
        json.dump(hashes, f, indent=2)

def file_hash(path: Path) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


## PDF  beolvasás, chunkolás
Beolvasás: PyMuPDF (fitz)
  - Ha nem olvasható vagy üres a program kihagyja

Chunkolás:
  - LLM kontextusablakának támogatására a szöveget kisebb részekre bontjuk
  - Pontosabb szemantikus keresés
Paraméterek:
  - chunk size: 500 szó. Ez nagyjából megfelel egy átlagos fogalom magyarázatnak. Ennél kisebb(100-200) túl kevés kontextust adna, nagyobb (pl. 1000) rontaná a keresést.
  - overlap: 50 szó. 10%-os átfedés két chunk között. Elegendő a folytonossághoz, adat duplázás nélkül.

Validáció:
  - Csak pdf fileokat fogad el a program.

Indexelés: Minden chunk ChromaDB-be kerül

Metaadatok:
  - subject: tárgy neve
  - source: melyik pdf fileból származik az információ.
  - chunk_index: hányadik a fileon belül (<- Jövőbeli funkció megvalósításához. Felhasználó például az 'X' PDF, első fejezetéből akar kérdezni)


In [16]:

def extract_text_from_pdf(pdf_path: Path) -> str:
    try:
        doc = fitz.open(str(pdf_path))
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text.strip()
    except Exception as e:
        raise ValueError(f"PDF olvasási hiba ({pdf_path.name}): {e}")


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

def embed_chunks(chunks: list[str]) -> list[list[float]]:
    return embedding_model.encode(chunks, show_progress_bar=False).tolist()

def validate_folder(subject_path: Path) -> list[Path]:
    all_files = list(subject_path.iterdir())
    non_pdfs = [f for f in all_files if f.is_file() and f.suffix.lower() != ".pdf"]

    if non_pdfs:
        raise TypeError(
            f"Nem PDF fájl(ok) találhatók a '{subject_path.name}' mappában: "
            f"{[f.name for f in non_pdfs]}\n"
        )

    return [f for f in all_files if f.is_file() and f.suffix.lower() == ".pdf"]

def index_all_subjects():
    subjects = [f for f in DOCS_DIR.iterdir() if f.is_dir()]

    if not subjects:
        print(" Nincs tárgy mappa a docs/ könyvtárban.")
        print(f"Hozz létre almappákat itt: {DOCS_DIR.resolve()}")
        return

    hashes = load_hashes()
    total_new = 0

    for subject_path in subjects:
        subject_name = subject_path.name
        print(f"\n Tárgy: {subject_name}")

        try:
            pdf_files = validate_folder(subject_path)
        except TypeError as e:
            print(e)
            continue

        if not pdf_files:
            print(f"   Nincs PDF fájl a '{subject_name}' mappában, kihagyva.")
            continue

        collection = chroma_client.get_or_create_collection(
            name=subject_name,
            metadata={"subject": subject_name}
        )

        new_files = 0
        for pdf_path in pdf_files:
            current_hash = file_hash(pdf_path)
            hash_key = str(pdf_path)

            if hashes.get(hash_key) == current_hash:
                print(f"   Már indexelve (változatlan): {pdf_path.name}")
                continue

            print(f"   Indexelés: {pdf_path.name}...")

            try:
                text = extract_text_from_pdf(pdf_path)
            except ValueError as e:
                print(f"{e}")
                continue

            if not text:
                print(f"   Üres PDF, kihagyva: {pdf_path.name}")
                continue

            chunks = chunk_text(text)
            embeddings = embed_chunks(chunks)
            ids = [f"{subject_name}__{pdf_path.stem}__{i}" for i in range(len(chunks))]

            try:
                existing_ids = collection.get(where={"source": pdf_path.name})["ids"]
                if existing_ids:
                    collection.delete(ids=existing_ids)
            except:
                pass

            collection.add(
                ids=ids,
                embeddings=embeddings,
                documents=chunks,
                metadatas=[{
                    "subject": subject_name,
                    "source": pdf_path.name,
                    "chunk_index": i
                } for i in range(len(chunks))]
            )

            hashes[hash_key] = current_hash
            new_files += 1
            total_new += 1
            print(f"   {pdf_path.name} – {len(chunks)} chunk indexelve.")

        print(f"   {subject_name}: {new_files} új fájl indexelve.")

    save_hashes(hashes)

    print(f"\n{'='*50}")
    print(f"Indexelés kész. Összesen {total_new} új fájl.")
    print(f"Kollekciók: {[c.name for c in chroma_client.list_collections()]}")

index_all_subjects()


 Tárgy: Biologia
   Már indexelve (változatlan): FI-505031101_Biologia11tk_2016_NKP.pdf
   Biologia: 0 új fájl indexelve.

 Tárgy: Matematika
   Indexelés: obadovics_matematika.pdf...
   Üres PDF, kihagyva: obadovics_matematika.pdf
   Már indexelve (változatlan): OH-MAT11TA__teljes.pdf
   Matematika: 0 új fájl indexelve.

Indexelés kész. Összesen 0 új fájl.
Kollekciók: ['Matematika', 'Biologia']


## Search engine inicializálása
Tavily:
  - Ingyenes tier: 1000 keresés/hó, elegendő prototípushoz
  - LangChain natív integráció, egy sorban használható
  - Google Search API fizetős és bonyolultabb a beállítása

In [17]:
from langchain_tavily import TavilySearch

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

search_tool = TavilySearch(
    max_results=3,
    api_key=TAVILY_API_KEY
)

# LangGraph Pipeline – Agentic RAG
## AppState
A LangGraph `TypedDict` alapú state-ben tárolja a pipeline összes köztes értékét:
- `messages` – teljes beszélgetés history, az `add_messages` annotáció automatikusan kezeli a hozzáfűzést
- `current_input` – aktuális felhasználói üzenet
- `task_type` – azonosított feladat (QA / SUMMARIZE / STUDY_PLAN / CLARIFY)
- `retrieved_chunks` – ChromaDB-ből visszakeresett szövegrészletek
- `web_search_needed` – az evaluator döntése hogy kell-e web keresés
- `web_search_results` – Tavily találatok
- `satisfactory` – az önértékelő node döntése hogy elég jó-e a válasz
- `retry_count` – hányszor próbálta már újra (max 2)

### Node-ok és a pipeline folyamata
        
![image.png](graph.png)

**`process_input`** – reseteli a state köztes értékeit minden új kérdésnél.

**`task_classifier`** – LLM dönti el mit akar a felhasználó. Ha nem egyértelmű az input, `CLARIFY` taskot rendel hozzá és visszakérdez.

**`rag_retriever`** – szemantikus keresés a ChromaDB-ben. Ha több tárgyból is jönnek találatok, `_MULTI` flag-et kap a task type, és a válaszban jelzi melyik tárgyból származik az info.

**`evaluator`** – LLM értékeli a visszakeresett chunkokat. Ha nem relevánsak a kérdéshez, web keresést indít.

**`web_search`** – Tavily keresés, csak akkor fut ha az evaluator gyengének ítélte a RAG találatokat.

**`llm_node`** – generálja a választ a kontextus alapján. Ha volt web keresés, csak azt használja; ha nem, a RAG chunkokat.

**`postprocess`** – formázza a végső választ. Tanulási terv esetén instrukciót fűz hozzá a folytatáshoz.

**`self_eval_node`** – **ReAct loop**: az LLM megvizsgálja saját válaszát. Ha nem elég jó, átfogalmazza a keresési kifejezést és visszamegy a `rag_retriever`-hez. Maximum 2 újrapróbálkozás engedélyezett.

In [18]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
import json

class AppState(TypedDict):
    messages: Annotated[list, add_messages]
    current_input: str
    retry_count: int
    revised_query: str
    retrieved_chunks: list[dict]
    detected_subjects: list[str]

    web_search_needed: bool
    web_search_results: list[dict]
    skip_rag: bool

    task_type: str
    ambiguity_detected: bool
    clarification_question: str

    final_response: str
    satisfactory: bool

def retrieve_chunks(query: str, n_results: int = 5) -> list[dict]:
    query_embedding = embedding_model.encode(query).tolist()
    collections = chroma_client.list_collections()

    if not collections:
        return []

    all_results = []
    for col in collections:
        collection = chroma_client.get_collection(col.name)
        try:
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=min(n_results, collection.count()),
                include=["documents", "metadatas", "distances"]
            )
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]
            ):
                all_results.append({
                    "text": doc,
                    "subject": meta.get("subject", "ismeretlen"),
                    "source": meta.get("source", "ismeretlen"),
                    "distance": dist
                })
        except:
            continue

    all_results.sort(key=lambda x: x["distance"])
    return all_results[:n_results]

def format_chunks_for_prompt(chunks: list[dict], max_chars: int = 500) -> str:
    if not chunks:
        return "Nem található releváns anyag."

    formatted = []
    for i, chunk in enumerate(chunks):
        text = chunk['text'][:max_chars]
        if len(chunk['text']) > max_chars:
            text += "..."
        formatted.append(
            f"[{i+1}. forrás – Tárgy: {chunk['subject']}, Fájl: {chunk['source']}]\n{text}"
        )
    return "\n\n".join(formatted)

def format_web_results_for_prompt(results: list[dict], max_chars: int = 500) -> str:
    if not results:
        return "Nem található webes találat."
    formatted = []
    for i, r in enumerate(results):
        # Levágjuk a contentet max_chars karakterre (különben nem fér el)
        content = r.get("content", "")[:max_chars]
        formatted.append(content)

    return "\n\n".join(formatted)
def format_history_for_prompt(messages: list) -> str:
    if not messages:
        return "Nincs korábbi beszélgetés."

    history = []
    for msg in messages[-6:]:
        role = "Felhasználó" if msg.type == "human" else "Asszisztens"
        history.append(f"{role}: {msg.content}")
    return "\n".join(history)

def process_input(state: AppState) -> AppState:
    print("\n" + "="*50)
    print("Input feldolgozása")
    print("="*50)
    state["retry_count"] = 0
    state["retrieved_chunks"] = []
    state["detected_subjects"] = []
    state["task_type"] = ""
    state["ambiguity_detected"] = False
    state["clarification_question"] = ""
    state["web_search_needed"] = False
    state["web_search_results"] = []
    state["final_response"] = ""

    return state


def task_classifier(state: AppState) -> AppState:
    print("\n" + "="*50)
    print("Task azonosítása")

    history = format_history_for_prompt(state["messages"])

    prompt = f"""
Te egy tanulást segítő asszisztens vagy. Elemezd a felhasználó üzenetét és döntsd el:

1. Milyen feladatot kér? Válassz egyet:
   - QA: kérdésre vár választ az anyagból
   - SUMMARIZE: összefoglalót kér
   - STUDY_PLAN: tanulási tervet kér
   - CLARIFY: nem egyértelmű, vissza kell kérdezni

2. Egyértelmű-e az utasítás? (igen/nem)
3. Ha nem egyértelmű, mit kérdezz vissza?
4. Kell-e dokumentumkeresés? skip_rag csak akkor true ha a kérdés:
   - Köszönés vagy általános csevegés (pl. "szia", "hogy vagy")
   - Az alkalmazás képességeire kérdez (pl. "mit tudsz csinálni?")
   - Egyébként MINDIG false, akkor is ha általános tudományos kérdés
Korábbi beszélgetés:
{history}

Jelenlegi üzenet: {state["current_input"]}

Válaszolj CSAK ebben a JSON formátumban:
{{
  "task_type": "QA|SUMMARIZE|STUDY_PLAN|CLARIFY",
  "ambiguous": true/false,
  "clarification_question": "kérdés vagy üres string",
  "web_search_needed": true/false
  "skip_rag": true/false
}}
"""
    response = invoke_llm(prompt)

    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        parsed = json.loads(response[start:end])

        state["task_type"] = parsed.get("task_type", "QA")
        state["ambiguity_detected"] = parsed.get("ambiguous", False)
        state["clarification_question"] = parsed.get("clarification_question", "")
        state["skip_rag"] = parsed.get("skip_rag", False)
    except:
        state["task_type"] = "QA"
        state["ambiguity_detected"] = False
        state["skip_rag"] = False
    if not chroma_client.list_collections():
        state['skip_rag'] = True
    print(f"  Task típus: {state['task_type']}, Kétértelmű: {state['ambiguity_detected']}")
    print(f"  Skip RAG: {state['skip_rag']}")
    print("="*50)
    return state

def clarify_node(state: AppState) -> AppState:
    print("Visszakérdezés")

    question = state["clarification_question"] or \
        "Pontosítanád kérlek, hogy mit szeretnél? Összefoglalót, kérdésre választ, vagy tanulási tervet kérsz?"

    state["final_response"] = question
    return state

def rag_retriever(state: AppState) -> AppState:
    print("RAG keresés")

    chunks = retrieve_chunks(state["current_input"], n_results=6)
    state["retrieved_chunks"] = chunks

    subjects = list(set(c["subject"] for c in chunks))
    state["detected_subjects"] = subjects

    print(f"  Talált tárgyak: {subjects}")
    print(f"  Visszakeresett chunkok: {len(chunks)}")

    if len(subjects) > 1:
        state["task_type"] = state["task_type"] + "_MULTI"

    return state

def self_evaluation_node(state: AppState) -> AppState:
    print("Önértékelés...")
    answer = state["final_response"]
    prompt = f"""
                Értékeld a saját válaszodat:

                Kérdés: {state["current_input"]}
                Válasz: {state["final_response"]}

                Ha a válasz nem elég részletes vagy nem válaszolja meg a kérdést, fogalmazd át a keresési kifejezést.

                Válaszolj CSAK ebben a JSON formátumban:
                {{
                "satisfactory": true/false,
                "revised_query": "átfogalmazott keresés vagy üres string"
                }}
            """
    response = invoke_llm(prompt)

    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        parsed = json.loads(response[start:end])
        state["satisfactory"] = parsed.get("satisfactory", True)
        state["revised_query"] = parsed.get("revised_query", "")
        if not state["satisfactory"]:
            state["retry_count"] += 1
            state["current_input"] = state["revised_query"]
    except:
        state["satisfactory"] = True

    return state
def route_after_self_eval(state: AppState) -> str:
    if not state["satisfactory"] and state["retry_count"] <= 2:
        return "rag"
    return "end"
def evaluator_node(state: AppState) -> AppState:
    print("Találatok értékelése")
    chunks = state["retrieved_chunks"]
    context_preview = "\n".join([c["text"][:200] for c in chunks[:3]])
    prompt = f"""
                Értékeld hogy az alábbi szövegrészletek relevánsak-e a kérdés megválaszolásához.

                Kérdés: {state["current_input"]}

                Visszakeresett szövegrészletek:
                {context_preview}

                Válaszolj CSAK ebben a JSON formátumban:
                {{
                "relevant": true/false,
                "reason": "rövid indoklás"
                }}
            """
    response = invoke_llm(prompt)

    try:
        start = response.find("{")
        end = response.rfind("}") + 1
        parsed = json.loads(response[start:end])
        relevant = parsed.get("relevant", False)
        reason = parsed.get("reason", "")

        print(f"LLM értékelés: releváns={relevant}, ok={reason}")

        state["web_search_needed"] = not relevant

    except:
        state["web_search_needed"] = False

    return state
def web_search_node(state: AppState) -> AppState:
    print("Web keresés...")

    try:
        results = search_tool.invoke({"query": state["current_input"]})
        if isinstance(results, list):
            state["web_search_results"] = results # Tavily listben ad vissza
        else:
            state["web_search_results"] = [{"content": str(results), "url": ""}]
        print(f"  Web találatok: {len(state['web_search_results'])} db")
    except Exception as e:
        print(f"  Web keresés hiba: {e}")
        state["web_search_results"] = []

    return state
def llm_node(state: AppState) -> AppState:
    print("LLM válasz generálása...")
    has_web = bool(state["web_search_results"])
    if has_web:
        context = format_web_results_for_prompt(state["web_search_results"])
    else:
        context = format_chunks_for_prompt(state["retrieved_chunks"])

    history = format_history_for_prompt(state["messages"])
    task = state["task_type"].replace("_MULTI", "")

    if task == "SUMMARIZE":
        task_instruction = "Készíts részletes összefoglalót az alábbi anyag alapján."
    elif task == "STUDY_PLAN":
        task_instruction = """Készíts 3 lépéses tanulási tervet:
                                1. lépés: Elméleti összefoglaló
                                2. lépés: Kulcsfogalmak és példák
                                3. lépés: Ellenőrző kérdések / mini teszt
                            """
    else:
        task_instruction = "Válaszolj a felhasználó kérdésére az alábbi anyag alapján."

    multi_subject_warning = ""
    if "_MULTI" in state["task_type"]:
        if state["web_search_needed"] is False:
            multi_subject_warning = f"""
                                Megjegyzés: A kérdéshez több tárgyból is találtam releváns anyagot: {state['detected_subjects']}
                                Jelezd a válaszodban melyik tárgyból származik az információ.
                                Ha úgy látod hogy egy anyag rossz mappában van, jelezd ezt is.
                                """
    prompt = f"""
                Te egy tanulást segítő asszisztens vagy. Feladatod: {task_instruction}

                Korábbi beszélgetés:
                    {history}

                Releváns anyagok:
                    {context}
                    {multi_subject_warning}
                Felhasználó kérése:
                    {state["current_input"]}

                Válaszolj magyarul, pontosan és érthetően.
"""
    response = invoke_llm(prompt)
    state["final_response"] = response
    return state


def postprocess(state: AppState) -> AppState:
    print("Postprocesszálás...")
    response = state["final_response"]
    if "STUDY_PLAN" in state["task_type"]:
        response += "\n\n---\nMelyik lépést szeretnéd részletesebben? Írj rá és folytatjuk!"

    state["final_response"] = response
    return state

def route_after_classifier(state: AppState) -> str:
    if state["ambiguity_detected"]:
        return "clarify"
    if state["skip_rag"]:
        return "llm"
    return "rag"

def route_after_rag(state: AppState) -> str:
    if not state["retrieved_chunks"]:
        return "no_results"
    return "evaluator"
def no_results_node(state: AppState) -> AppState:
        state["final_response"] = (
            "Nem találtam releváns anyagot a feltett kérdéshez. "
            "Győződj meg róla hogy a megfelelő PDF-ek indexelve vannak a docs/ mappában."
        )
        return state
def route_after_evaluator(state: AppState) -> str:
    if state["web_search_needed"]:
        return "web_search"
    return "llm"
def build_graph() -> StateGraph:
    graph = StateGraph(AppState)
    graph.add_node("process_input", process_input)
    graph.add_node("task_classifier", task_classifier)
    graph.add_node("clarify", clarify_node)
    graph.add_node("evaluator", evaluator_node)
    graph.add_node("web_search", web_search_node)
    graph.add_node("rag_retriever", rag_retriever)
    graph.add_node("llm_node", llm_node)
    graph.add_node("postprocess", postprocess)
    graph.add_node("no_results", no_results_node)
    graph.set_entry_point("process_input")
    graph.add_edge("process_input", "task_classifier")
    graph.add_node("self_eval_node", self_evaluation_node)
    graph.add_conditional_edges(
        "task_classifier",
        route_after_classifier,
        {"clarify": "clarify", "rag": "rag_retriever", "llm": "llm_node"}
    )

    graph.add_conditional_edges(
        "rag_retriever",
        route_after_rag,
        {"evaluator": "evaluator", "no_results": "no_results"}
    )
    graph.add_conditional_edges(
       "self_eval_node",
        route_after_self_eval,
        {"rag": "rag_retriever", "end": END}
    )

    graph.add_conditional_edges(
        "evaluator",
        route_after_evaluator,
        {"web_search": "web_search", "llm": "llm_node"}
    )

    graph.add_edge("web_search", "llm_node")
    graph.add_edge("clarify", END)
    graph.add_edge("llm_node", "postprocess")
    graph.add_edge("no_results", END)
    graph.add_edge("postprocess", "self_eval_node")


    return graph.compile()


app_graph = build_graph()


### ChatSession osztály
A `ChatSession` osztály kezeli a teljes beszélgetés életciklusát:
- **state** – tárolja a LangGraph pipeline state-jét sessionök között, beleértve a teljes üzenet history-t
- **chat()** – egy kérdés-válasz kör: hozzáfűzi a user üzenetet, lefuttatja a pipeline-t, visszaadja a választ
- **reset()** – törli a history-t és nullázza a state-et, új session kezdődik
- **show_history()** – kiírja a teljes eddigi beszélgetést
- **show_subjects()** – listázza az indexelt tárgyakat és chunk számukat


In [19]:
from langchain_core.messages import HumanMessage, AIMessage
class ChatSession:
    def __init__(self):
        self.state: AppState = {
            "messages": [],
            "current_input": "",
            "retrieved_chunks": [],
            "detected_subjects": [],
            "task_type": "",
            "ambiguity_detected": False,
            "clarification_question": "",
            "retry_count": 0,
            "revised_query": "",
            "web_search_needed": False,
            "web_search_results": [],
            "final_response": "",
            "satisfactory": True,
        }

    def chat(self, user_input: str) -> str:
        self.state["messages"].append(HumanMessage(content=user_input))
        self.state["current_input"] = user_input

        self.state = app_graph.invoke(self.state)

        response = self.state["final_response"]
        self.state["messages"].append(AIMessage(content=response))

        return response

    def reset(self):
        self.__init__()
        print("Session törölve.")

    def show_history(self):
        if not self.state["messages"]:
            print("Nincs korábbi üzenet.")
            return
        print("\n" + "="*50)
        print("BESZÉLGETÉS HISTORY")
        print("="*50)
        for msg in self.state["messages"]:
            role = "Te" if isinstance(msg, HumanMessage) else "Asszisztens"
            print(f"\n{role}:\n{msg.content}")
        print("="*50 + "\n")

    def show_subjects(self):
        collections = chroma_client.list_collections()
        if not collections:
            print("Nincs indexelt tárgy.")
            return
        print("\nIndexelt tárgyak:")
        for col in collections:
            collection = chroma_client.get_collection(col.name)
            print(f"  - {col.name} ({collection.count()} chunk)")


def run_chat():
    session = ChatSession()
    print("="*50)
    print("Parancsok:")
    print(" 'kilép':  chat vége")
    print(" 'reset':  új session indítása")
    print(" 'history':korábbi üzenetek")
    print(" 'tárgyak':indexelt tárgyak listája")
    print("="*50 + "\n")

    session.show_subjects()
    print()

    while True:
        try:
            user_input = input("User: ").strip()
            print("User:", user_input)
        except (EOFError, KeyboardInterrupt):
            break

        if not user_input:
            continue

        if user_input.lower() == "kilép":
            break
        elif user_input.lower() == "reset":
            session.reset()
            continue
        elif user_input.lower() == "history":
            session.show_history()
            continue
        elif user_input.lower() == "tárgyak":
            session.show_subjects()
            continue

        print()
        response = session.chat(user_input)
        print(f"\nAsszisztens:\n{response}\n")
        print("-"*50)


In [21]:
run_chat()

Parancsok:
 'kilép':  chat vége
 'reset':  új session indítása
 'history':korábbi üzenetek
 'tárgyak':indexelt tárgyak listája


Indexelt tárgyak:
  - Matematika (249 chunk)
  - Biologia (187 chunk)

User: szia


Input feldolgozása

Task azonosítása
  Task típus: CLARIFY, Kétértelmű: False
  Skip RAG: True
LLM válasz generálása...
Postprocesszálás...
Önértékelés...

Asszisztens:
Szia! Miben segíthetek ma?

--------------------------------------------------
User: mi az a sejt?


Input feldolgozása

Task azonosítása
  Task típus: QA, Kétértelmű: False
  Skip RAG: False
RAG keresés
  Talált tárgyak: ['Biologia']
  Visszakeresett chunkok: 6
Találatok értékelése
LLM értékelés: releváns=True, ok=A szövegrészletek leírják a sejt szerkezetét, funkcióit és alapvető részeit, ami közvetlenül segíti a "mi az a sejt" kérdés megválaszolását.
LLM válasz generálása...
Postprocesszálás...
Önértékelés...

Asszisztens:
A sejt az élő szervezetek legkisebb, önállóan működő egysége, amely a biológiai élet a